# Jute Notebook Deck Mode

Code-graph analysis · 2026-05-28

*This deck IS a notebook. There is no sidecar slide file.*

## The model

## One cell. One slide.

- `.ipynb` is the **single source of truth** — no sidecar files
- Per-cell knobs live in `cell.metadata.jute_deck`
- Deck-wide knobs live in `notebook.metadata.jute_deck`
- V1 is **view-only** — no PDF / PPTX / HTML export
- Round-trips through any Jupyter tool unchanged

## By the numbers

## Surface area

- **22 files** across `ui/deck/`, `agent/deck/`, `PresentPage.tsx`, `set_cell_metadata.rs`
- **51 symbols** — types, helpers, components, MCP tool
- **841 indexed lines** total
- **5 commits** from design → merge
- Hottest symbol: `PresentPage` (121 lines), `BulletsLayout` (47), `CodeLayout` (34), `TitleLayout` / `SectionLayout` (30/30)

## Data model

## `cell.metadata.jute_deck` — six keys

- `layout` — `auto | title | section | content | bullets | code | output | code-output | two-col | image | blank`
- `hidden` — skip in deck mode, keep in notebook
- `speaker_notes` — markdown, shown only via the **S** key
- `theme_override` — per-slide theme id
- `fragments` — bullet-by-bullet reveal
- `background` — CSS color or image URL

## `notebook.metadata.jute_deck` — four keys

- `theme` — deck-wide default (currently inert — see slide 13)
- `aspect` — e.g. `"16:9"`
- `title` — deck title override (defaults to filename)
- `author` — display name

## Layout inference

## Inference rules — first match wins

- code cell → **output** (executed output is the slide)
- raw / html cell → **blank**
- lone `# H1` → **title**
- lone `## H2` → **section**
- any `# H1` anywhere → **title**
- has bullets → **bullets**
- everything else → **content**

## Render pipeline

## Single integration point

```
PresentPage (cellIds → cells)
    │
    ▼  cellToSlide(cell, deck?)
       │  inferLayout → buildBlocks → resolveTheme
       ▼  SlideSpec { id, layout, blocks, theme, ... }
          │
          ▼  <SlideFrame themeId background>
             <LayoutFor slide fragmentIndex />
```

**`cellToSlide` is called from exactly one site** — `PresentPage`. Clean blast radius.

## Mutation: the three-layer MCP pattern

## `notebook.set_cell_metadata` — three layers, one invariant

1. **Rust MCP tool** (`set_cell_metadata.rs`) — validates `expected_version >= 1`, forwards to bridge
2. **TS agent handler** (`handlers.ts`) — version check → merge, **no await between**
3. **zustand+immer store** (`stores/notebook.ts`) — `mergeCellJuteDeckMetadata` bumps version

Caller protocol: `read version → call → if conflict, re-read and retry`.

## Present mode

## Keyboard map

- `→` / `Space` / `PgDn` — next fragment, else next slide
- `←` / `PgUp` — previous fragment, else previous slide
- `Home` / `End` — first / last slide
- `S` — toggle speaker-notes overlay
- `B` — toggle blackout
- `Esc` — back to `/notebook?path=…`

## Themes

## Three presets — closed enum

- `minimal-light` (default) — white bg, slate-900 text
- `minimal-dark` — slate-900 bg, slate-50 text
- `spur-brand` — indigo→violet gradient, amber accent

Unknown ids fall back to `minimal-light` via `resolveTheme`. To customize, add to `THEMES` in `themes.ts`.

## How it shipped

## Lineage — five commits

- `20f2f889` design spec — notebook-native presentations
- `28b964d9` scope down to view-only (drop export)
- `c03d782c` cross-map design against real code graph
- `3bbbffa7` 13-task implementation plan, 6 phases
- `7bc4efd3` merge — 12 tasks landed (T13 verification-only)

## Anti-patterns

## Don't do these

- `write_cell` + `set_cell_metadata` without re-reading version
- Setting `layout` on every cell — let inference do its job
- Using `hidden: true` as a comment-out for drafts
- Treating `notebook.metadata.jute_deck.theme` as live (v1: inert)
- Speaker notes that repeat the visible slide
- `fragments: true` on non-bullet slides — no-op
- Producing a separate `.md` / `.json` deck file
- Trying to export to PDF / PPTX — no path exists in v1

## TL;DR

**The notebook IS the deck.** One cell, one slide. `cell.metadata.jute_deck` controls how a slide renders; never how many slides exist. Every `set_cell_metadata` carries `expected_version` and retries on conflict. Layout inference is well-defined — write good markdown and override only for `two-col` / `image` / `code-output`. Present mode: `Cmd+Shift+P → Deck → Enter present mode`.